# Лабораторная работа №5

### Методы интегрирования

Что было сделано в рамках лабораторной работы:
- Реализованы  методы численного интегрирования (метод прямоугольников, трапеций, симпсона, квадратур Гаусса, Монте-Карло) ([интеграторы](../src/comp_math/integration/impl/))
- Написаны тесты для проверки корректности интеграторов ([тесты](../tests/test_integrators/test_numerical_integration.py))

In [33]:
# Необходимые импорты
import numpy as np
from comp_math.integration.integrator_registry import IntegratorRegistry
import matplotlib.pyplot as plt
import pandas as pd

1) Интегриования функции из VII 9.3

In [34]:
x_data = np.array([-1.0, -0.75, -0.5, -0.25, 0.0, 0.25, 0.5, 0.75, 1.0])
f_data = np.array([-0.333, 0.0, -0.125, -0.056, 0.0, 0.046, 0.083, 0.115, 0.143])

Используемые методы

In [35]:
methods = [
    (IntegratorRegistry.create_solver("rectangle"), "Метод Прямоугольников"),
    (IntegratorRegistry.create_solver("trapezoida"), "Метод Трапеций"),
    (IntegratorRegistry.create_solver("simpson"), "Метод Симпсона"),
    (IntegratorRegistry.create_solver("gauss"), "Метод квадратур Гаусса"),
    (IntegratorRegistry.create_solver("monte_carlo"), "Метод Монте-Карло")
]

In [36]:
results = {}
for solver, name in methods:
    try:
        result = solver.integrate_table(x_data, f_data)
        results[name] = result
        
        # Для метода трапеций дополнительно вычисляем погрешность по Рунге
        if "Трапеций" in name:
            # Оценка погрешности по правилу Рунге
            x_half = x_data[::2]
            f_half = f_data[::2]
            result_half = solver.integrate_table(x_half, f_half)
            
            runge_error = abs(result - result_half) / 3
            
            print(f"Оценка погрешности по правилу Рунге: {runge_error:.6e}")
        
        print()
        
    except Exception as e:
        print(f"Ошибка при вычислении {name}: {e}")
        print()

for name, result in results.items():
    print(f"{name}: {result:.6f}")


Оценка погрешности по правилу Рунге: 7.000700e-06


Ошибка при вычислении Метод квадратур Гаусса: Метод Гаусса не поддерживает интегрирование по табличным данным напрямую. Нужно для этого использовать интерполяции.


Метод Прямоугольников: -0.000025
Метод Трапеций: -0.000006
Метод Симпсона: 0.012167
Метод Монте-Карло: -0.028222


Для интегратора методом Гаусса создадим функцию с помощью интерполятора

In [37]:
from comp_math.interpolation.impl.lsq_interpolator import UniversalLSQ

basis_funcs = [
    lambda x: 1,
    lambda x: x,
    lambda x: x**2,
    lambda x: x**3,
    lambda x: x**4
]

interpolator = UniversalLSQ(basis_functions=basis_funcs)

interpolator.fit(x_data, f_data)

def func_interpolated(x):
    """Функция на основе интерполяции"""
    return interpolator(np.array([x])).to_numpy()

Повторим вычисления для методов Гаусса и Монте-Карло с использованием функции, полученной интерполированием

In [38]:
# Для метода Гаусса используем интерполированную функцию
gauss_solver = methods[3][0]
result_gauss = gauss_solver.integrate_func(func_interpolated, -1, 1)
print(f"{methods[3][1]}: {result_gauss:.6f} (использована интерполяция)")

# Для метода Монте-Карло также используем интерполированную функцию
mc_solver = methods[4][0]
result_mc = mc_solver.integrate_func(func_interpolated, -1, 1)
print(f"{methods[4][1]}: {result_mc:.6f} (использована интерполяция)")

Метод квадратур Гаусса: -0.001409 (использована интерполяция)
Метод Монте-Карло: -0.001224 (использована интерполяция)
